# Statistics for Data Analysis Bootcamp

A practical notebook covering **32 statistics concepts** used in data analysis, experimentation, and business reporting. Each topic has exactly two cells: a Markdown explanation and a Python code cell with inline `#` comments explaining the steps.

**Requirements:** NumPy, pandas, SciPy, and Matplotlib. These are commonly available in Anaconda. If needed, run `%pip install numpy pandas scipy matplotlib` in a notebook cell.

The examples use synthetic and reproducible data; no external files are required. Run the notebook from top to bottom.

## Learning path
1. Describe data and distributions
2. Probability, sampling, and uncertainty
3. Confidence intervals and hypothesis tests
4. Compare groups, rates, and categorical variables
5. Correlation, regression, and bootstrap methods
6. A/B testing, power, and reporting pitfalls

**Important:** Statistical tests depend on assumptions and study design. Treat the examples as learning demonstrations, not automatic decision rules.

## 1. Descriptive Statistics: First Summary

Descriptive statistics summarize what a dataset contains before drawing conclusions. Common summaries include count, mean, median, minimum, maximum, standard deviation, and quartiles.

The example creates a synthetic sales dataset that later lessons reuse. These are illustrative values, not real business data. Use `describe()` for a quick overview, then investigate data types, missing values, and unexpected values separately.

In [ ]:
import numpy as np  # Import NumPy for numerical calculations.
import pandas as pd  # Import pandas for tabular data.
from scipy import stats  # Import SciPy's statistical methods.
import matplotlib.pyplot as plt  # Import Matplotlib for a few statistical plots.

rng = np.random.default_rng(42)  # Create a reproducible random-number generator.
sales_df = pd.DataFrame({  # Create an illustrative dataset with one row per transaction.
    "revenue": rng.gamma(shape=2.5, scale=120, size=250).round(2),  # Generate positive, right-skewed revenue values.
    "units": rng.integers(1, 8, size=250),  # Generate quantities from 1 through 7.
    "region": rng.choice(["North", "South", "East", "West"], size=250),  # Assign a region to each transaction.
    "channel": rng.choice(["Online", "Store", "Partner"], size=250),  # Assign a sales channel.
})  # Finish creating the sample dataset.
print(sales_df.head())  # Preview the first five rows.
print("\nDescriptive summary:")  # Add a label before the summary.
print(sales_df[["revenue", "units"]].describe().round(2))  # Summarize count, mean, standard deviation, quartiles, and range.

## 2. Mean, Median, and When to Use Them

The **mean** is the sum of observations divided by their count. The **median** is the middle value after sorting. The mean responds strongly to extreme values; the median is more robust to skew and outliers.

For skewed transaction values, report both. A mean much larger than the median often indicates a right-skewed distribution, although a plot and domain context should confirm the interpretation.

In [ ]:
values = np.array([20, 22, 24, 26, 28, 30, 32, 200])  # Create values with one unusually large observation.
mean_value = np.mean(values)  # Calculate the arithmetic mean.
median_value = np.median(values)  # Calculate the median.
without_extreme_mean = np.mean(values[:-1])  # Calculate the mean after excluding the final value for demonstration.
print("Mean with extreme value:", round(mean_value, 2))  # Show how the extreme value affects the mean.
print("Median with extreme value:", round(median_value, 2))  # Show the median, which is less affected by the extreme value.
print("Mean without extreme value:", round(without_extreme_mean, 2))  # Compare with the mean of the first seven observations.

## 3. Mode and Frequency

The **mode** is the most frequent value or category. It can be useful for discrete measures and categorical fields such as product, region, or customer segment.

A dataset may have one mode, several modes, or no uniquely informative mode. For categorical data, frequencies and proportions are often more useful than a mean.

In [ ]:
channels = pd.Series(["Online", "Store", "Online", "Partner", "Online", "Store", "Partner", "Online"])  # Create example channel labels.
frequency = channels.value_counts()  # Count occurrences of each channel.
proportion = channels.value_counts(normalize=True)  # Calculate each channel's share of all records.
modes = channels.mode().tolist()  # Return the most frequent category or categories.
print("Frequency:")  # Label the raw count output.
print(frequency)  # Display the number of records in each category.
print("\nProportion:")  # Label the proportion output.
print(proportion.round(3))  # Display category shares as decimals.
print("\nMode(s):", modes)  # Display the most frequent channel label(s).

## 4. Range, Variance, and Standard Deviation

The **range** is maximum minus minimum. **Variance** summarizes average squared deviations from the mean; **standard deviation** is its square root and is expressed in the original units.

NumPy's default `var()` and `std()` use the population convention (`ddof=0`). For estimating variance from a sample, `ddof=1` is common. Be explicit about which convention you use.

In [ ]:
delivery_minutes = np.array([18, 20, 21, 23, 24, 26, 28, 30])  # Store sample delivery times in minutes.
value_range = delivery_minutes.max() - delivery_minutes.min()  # Calculate the distance between the maximum and minimum.
population_variance = np.var(delivery_minutes, ddof=0)  # Calculate population variance.
sample_variance = np.var(delivery_minutes, ddof=1)  # Calculate sample variance with Bessel's correction.
population_std = np.std(delivery_minutes, ddof=0)  # Calculate population standard deviation.
sample_std = np.std(delivery_minutes, ddof=1)  # Calculate sample standard deviation.
print("Range:", value_range)  # Display the full spread from minimum to maximum.
print("Population variance:", round(population_variance, 2))  # Display population variance.
print("Sample variance:", round(sample_variance, 2))  # Display sample variance.
print("Population standard deviation:", round(population_std, 2))  # Display population standard deviation.
print("Sample standard deviation:", round(sample_std, 2))  # Display sample standard deviation.

## 5. Percentiles, Quartiles, and IQR

Percentiles describe the position of observations in a distribution. The 25th percentile is Q1, the 50th is the median, and the 75th is Q3. The **interquartile range** is `IQR = Q3 − Q1`, representing the spread of the middle 50% of values.

Percentiles are useful for service-level reporting, such as the 90th percentile delivery time. Be aware that software may use slightly different interpolation methods for percentiles.

In [ ]:
delivery_days = np.array([1, 2, 2, 3, 3, 4, 4, 5, 7, 12, 15])  # Create delivery-time observations in days.
q1, median, q3 = np.percentile(delivery_days, [25, 50, 75])  # Calculate the first quartile, median, and third quartile.
iqr = q3 - q1  # Calculate the interquartile range.
p90 = np.percentile(delivery_days, 90)  # Calculate the 90th percentile.
print("Q1:", q1)  # Display the 25th percentile.
print("Median:", median)  # Display the 50th percentile.
print("Q3:", q3)  # Display the 75th percentile.
print("IQR:", iqr)  # Display the middle-50-percent spread.
print("90th percentile:", p90)  # Display the 90th-percentile delivery time.

## 6. Detecting Potential Outliers with IQR

A common screening rule flags values below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR`. This rule identifies **potential** outliers; it does not prove a value is erroneous.

Investigate unusual observations with business context. A large valid transaction should not be deleted simply because a statistical rule flags it.

In [ ]:
transaction_values = np.array([22, 24, 25, 27, 28, 29, 31, 33, 36, 210])  # Create sample transaction amounts with one extreme value.
q1, q3 = np.percentile(transaction_values, [25, 75])  # Calculate the first and third quartiles.
iqr = q3 - q1  # Measure the spread of the middle half.
lower_limit = q1 - 1.5 * iqr  # Calculate the lower screening threshold.
upper_limit = q3 + 1.5 * iqr  # Calculate the upper screening threshold.
outlier_mask = (transaction_values < lower_limit) | (transaction_values > upper_limit)  # Flag values outside the thresholds.
print("Lower limit:", round(lower_limit, 2))  # Display the lower boundary.
print("Upper limit:", round(upper_limit, 2))  # Display the upper boundary.
print("Potential outliers:", transaction_values[outlier_mask])  # Display values for investigation, without deleting them.

## 7. Z-Scores and Standardization

A **z-score** measures how many standard deviations a value lies above or below the mean: `z = (x − mean) / standard deviation`.

Z-scores make measurements on different scales easier to compare. A large absolute z-score can indicate an unusual observation, but a cutoff such as 3 is only a rule of thumb and depends on distribution assumptions.

In [ ]:
exam_scores = np.array([55, 60, 65, 70, 75, 80, 95], dtype=float)  # Store example exam scores.
mean_score = exam_scores.mean()  # Calculate the mean score.
std_score = exam_scores.std(ddof=0)  # Calculate the population standard deviation.
z_scores = (exam_scores - mean_score) / std_score  # Convert each score into standard-deviation units.
print("Mean:", round(mean_score, 2))  # Display the average score.
print("Standard deviation:", round(std_score, 2))  # Display the spread.
print("Z-scores:", z_scores.round(2))  # Display standardized values.
print("Values with |z| > 2:", exam_scores[np.abs(z_scores) > 2])  # Flag values more than two standard deviations from the mean for review.

## 8. Understanding Distribution Shape

A distribution describes how values are spread. Look for symmetry or skew, multiple peaks, long tails, and gaps. A histogram is useful, but results depend on bin selection.

Skewness is a numerical summary of asymmetry; kurtosis summarizes tail heaviness relative to a reference convention. Neither statistic replaces inspecting the distribution, especially for small samples.

In [ ]:
revenue_values = sales_df["revenue"].to_numpy()  # Convert sample revenue into a NumPy array.
skewness = stats.skew(revenue_values, bias=False)  # Estimate distribution asymmetry with a sample correction.
excess_kurtosis = stats.kurtosis(revenue_values, fisher=True, bias=False)  # Calculate excess kurtosis, where a normal distribution is near zero.
fig, ax = plt.subplots(figsize=(8, 4))  # Create a figure and axis for the distribution plot.
ax.hist(revenue_values, bins=20, edgecolor="white")  # Plot the frequencies of revenue values.
ax.set_title("Distribution of Sample Revenue")  # Add a clear chart title.
ax.set_xlabel("Revenue (illustrative units)")  # Label the measurement axis.
ax.set_ylabel("Count")  # Label the frequency axis.
fig.tight_layout()  # Adjust spacing so the labels fit.
plt.show()  # Display the histogram.
print("Skewness:", round(skewness, 2))  # Display the estimated asymmetry.
print("Excess kurtosis:", round(excess_kurtosis, 2))  # Display the estimated excess kurtosis.

## 9. Basic Probability

Probability quantifies how likely an event is, on a scale from 0 to 1. An empirical probability estimates this as the number of times an event occurs divided by the number of observed trials.

The empirical estimate depends on the observed sample. It is not a guarantee about the next observation, and estimates are more variable with fewer trials.

In [ ]:
rng_prob = np.random.default_rng(7)  # Create a separate reproducible random generator.
rolls = rng_prob.integers(1, 7, size=10000)  # Simulate 10,000 fair six-sided die rolls.
successes = np.sum(rolls >= 5)  # Count outcomes of 5 or 6.
empirical_probability = successes / len(rolls)  # Estimate the event probability from the simulated data.
theoretical_probability = 2 / 6  # Calculate the known probability of rolling 5 or 6.
print("Successful rolls:", successes)  # Display the observed success count.
print("Empirical probability:", round(empirical_probability, 4))  # Display the simulation-based estimate.
print("Theoretical probability:", round(theoretical_probability, 4))  # Display the exact probability for comparison.

## 10. Conditional Probability

Conditional probability asks how likely event A is among cases where event B has occurred: `P(A | B) = P(A and B) / P(B)`, provided `P(B) > 0`.

For analysis, a conditional rate often corresponds to a filtered denominator. For example, “purchase rate among users who added to cart” is not the same as “purchase rate among all visitors.”

In [ ]:
visitors = 1000  # Record the number of visitors in an illustrative funnel.
cart_users = 280  # Record visitors who added an item to the cart.
buyers = 120  # Record visitors who completed a purchase, assumed to be among cart users here.
p_cart_given_visitor = cart_users / visitors  # Calculate the share of visitors who added to cart.
p_purchase_given_cart = buyers / cart_users  # Calculate purchase probability among cart users.
p_purchase_given_visitor = buyers / visitors  # Calculate overall visitor-to-purchase conversion.
print("P(cart | visitor):", f"{p_cart_given_visitor:.1%}")  # Display cart conversion among visitors.
print("P(purchase | cart):", f"{p_purchase_given_cart:.1%}")  # Display purchase conversion among cart users.
print("P(purchase | visitor):", f"{p_purchase_given_visitor:.1%}")  # Display the end-to-end conversion rate.

## 11. Bayes' Theorem

Bayes' theorem updates a prior probability using evidence: `P(A | B) = P(B | A) × P(A) / P(B)`.

It is useful for understanding screening tests, fraud alerts, and classification outputs. The key lesson is that the probability an alert is correct depends not only on test accuracy but also on how common the event is in the first place.

In [ ]:
prevalence = 0.01  # Assume 1% of transactions are genuinely fraudulent.
sensitivity = 0.95  # Assume the detector flags 95% of fraudulent transactions.
false_positive_rate = 0.05  # Assume it incorrectly flags 5% of legitimate transactions.
true_positive_probability = sensitivity * prevalence  # Calculate the fraction of all transactions that are true positives.
false_positive_probability = false_positive_rate * (1 - prevalence)  # Calculate the fraction that are false positives.
probability_flag_is_fraud = true_positive_probability / (true_positive_probability + false_positive_probability)  # Apply Bayes' theorem to a flagged transaction.
print("Probability a flagged transaction is fraud:", f"{probability_flag_is_fraud:.1%}")  # Show the posterior probability under these assumptions.

## 12. Random Sampling and Sampling Bias

A sample is used to learn about a larger population. A simple random sample gives each eligible record an equal chance of selection, while stratified sampling ensures representation from specified groups.

A large sample can still be biased if the selection process systematically excludes part of the population. Sampling design and data collection quality matter as much as sample size.

In [ ]:
rng_sample = np.random.default_rng(123)  # Create a reproducible random generator for sampling.
customer_ids = np.arange(1, 101)  # Create 100 illustrative customer identifiers.
random_indices = rng_sample.choice(len(customer_ids), size=10, replace=False)  # Select ten distinct positions at random.
random_sample = customer_ids[random_indices]  # Retrieve the sampled customer IDs.
print("Random sample of customer IDs:", random_sample)  # Display the selected sample.
segments = np.repeat(["Consumer", "Business", "Small Business"], [60, 25, 15])  # Create a population with three different segment sizes.
segment_df = pd.DataFrame({"customer_id": customer_ids, "segment": segments})  # Align each customer with a segment.
stratified_sample = segment_df.groupby("segment", group_keys=False).sample(frac=0.2, random_state=123)  # Select 20% from each segment.
print("\nStratified sample counts:")  # Add a label for the group counts.
print(stratified_sample["segment"].value_counts())  # Verify that every segment is represented proportionally.

## 13. Standard Error

The **standard deviation** describes variation among observations. The **standard error (SE)** describes the uncertainty of a sample statistic across repeated samples.

For a sample mean, the usual estimated standard error is `sample standard deviation / sqrt(sample size)`. Larger independent samples typically produce a smaller standard error, but bias or dependence is not fixed by increasing sample size.

In [ ]:
rng_se = np.random.default_rng(8)  # Create a reproducible random generator.
population = rng_se.normal(loc=100, scale=20, size=100000)  # Simulate a population with mean 100 and standard deviation 20.
for n in [25, 100, 400]:  # Compare three sample sizes.
    sample = rng_se.choice(population, size=n, replace=False)  # Draw a random sample of the current size.
    standard_error = sample.std(ddof=1) / np.sqrt(n)  # Estimate the standard error of the sample mean.
    print(f"n={n:3d} | sample mean={sample.mean():.2f} | estimated SE={standard_error:.2f}")  # Display the mean and its estimated standard error.

## 14. Central Limit Theorem (CLT)

The Central Limit Theorem says that, under common conditions, the sampling distribution of the sample mean becomes approximately normal as sample size increases, even when the population itself is not normal.

The CLT concerns the distribution of a statistic across repeated samples—not the claim that the original data becomes normally distributed. Independence, finite variance, and sample size all matter.

In [ ]:
rng_clt = np.random.default_rng(9)  # Create a reproducible random generator.
population = rng_clt.exponential(scale=2.0, size=200000)  # Generate a strongly right-skewed population.
sample_size = 40  # Set the number of observations in each sample.
number_of_samples = 3000  # Set the number of repeated samples to simulate.
sample_means = np.array([rng_clt.choice(population, size=sample_size, replace=True).mean() for _ in range(number_of_samples)])  # Calculate one mean per repeated sample.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))  # Create panels for the population and sample means.
axes[0].hist(population[:5000], bins=35, edgecolor="white")  # Plot a subset of the skewed population values.
axes[0].set_title("Original Population (Subset)")  # Title the first panel.
axes[0].set_xlabel("Value")  # Label the first panel's x-axis.
axes[0].set_ylabel("Count")  # Label the first panel's y-axis.
axes[1].hist(sample_means, bins=35, edgecolor="white")  # Plot the repeated sample means.
axes[1].set_title("Sampling Distribution of the Mean")  # Title the second panel.
axes[1].set_xlabel("Sample mean")  # Label the second panel's x-axis.
axes[1].set_ylabel("Count")  # Label the second panel's y-axis.
fig.tight_layout()  # Adjust the panel layout.
plt.show()  # Display both histograms.
print("Population mean:", round(population.mean(), 2))  # Display the approximate population mean.
print("Mean of sample means:", round(sample_means.mean(), 2))  # Compare the average sample mean with the population mean.

## 15. Confidence Interval for a Mean

A confidence interval gives a range of values produced by a method that, over repeated samples, would cover the fixed population parameter at the stated rate (for example, 95%).

For a mean with unknown population standard deviation, a t interval uses the sample standard deviation and a t critical value. A 95% confidence interval does **not** mean there is a 95% probability that this already-computed interval contains the fixed mean under the frequentist interpretation.

In [ ]:
delivery = np.array([28, 31, 27, 35, 30, 29, 33, 32, 26, 34, 30, 31], dtype=float)  # Store a small sample of delivery times.
n = len(delivery)  # Count observations.
sample_mean = delivery.mean()  # Calculate the sample mean.
sample_se = stats.sem(delivery)  # Calculate the estimated standard error of the mean.
critical_t = stats.t.ppf(0.975, df=n - 1)  # Get the two-sided 95% t critical value.
margin = critical_t * sample_se  # Calculate the margin of error.
ci_low = sample_mean - margin  # Calculate the lower confidence limit.
ci_high = sample_mean + margin  # Calculate the upper confidence limit.
print("Sample mean:", round(sample_mean, 2))  # Display the point estimate.
print("95% t confidence interval:", (round(ci_low, 2), round(ci_high, 2)))  # Display the interval for the population mean.

## 16. Hypothesis Testing and P-Values

A hypothesis test evaluates data against a null hypothesis. The **p-value** is the probability, assuming the null model and its assumptions are true, of observing a result at least as extreme as the one obtained.

A p-value is not the probability that the null hypothesis is true, nor does it measure business importance. State the hypotheses, test, effect size, confidence interval, and practical context together.

In [ ]:
sample = np.array([52, 49, 55, 51, 53, 48, 54, 52, 50, 56], dtype=float)  # Store sample observations.
null_mean = 50  # Define the population mean under the null hypothesis.
test_result = stats.ttest_1samp(sample, popmean=null_mean)  # Run a two-sided one-sample t-test.
print("Sample mean:", round(sample.mean(), 2))  # Display the observed mean.
print("t statistic:", round(test_result.statistic, 3))  # Display the standardized test statistic.
print("p-value:", round(test_result.pvalue, 4))  # Display the probability of a result at least as extreme under the null model.
print("Interpretation: compare the p-value with a preselected alpha and consider effect size and assumptions.")  # Remind the analyst to interpret the result in context.

## 17. One-Sample t-Test

A one-sample t-test compares a sample mean with a specified reference value. It is commonly used when comparing an average measure against a benchmark, assuming observations are independent and the sample is appropriate for inference.

For small samples, severe skew or influential outliers can undermine the test. Consider plots and robust or non-parametric alternatives when assumptions are questionable.

In [ ]:
response_times = np.array([4.8, 5.1, 4.9, 5.3, 5.0, 4.7, 5.2, 5.1, 4.9, 5.4])  # Record sample response times in seconds.
benchmark = 5.0  # Set the reference mean to test.
result = stats.ttest_1samp(response_times, popmean=benchmark)  # Test whether the sample mean differs from the benchmark.
print("Sample mean:", round(response_times.mean(), 3))  # Display the observed average.
print("Benchmark:", benchmark)  # Display the reference value.
print("t statistic:", round(result.statistic, 3))  # Display the test statistic.
print("Two-sided p-value:", round(result.pvalue, 4))  # Display the p-value for a difference in either direction.

## 18. Independent Two-Sample t-Test

An independent two-sample t-test compares the means of two independent groups. Welch's t-test (`equal_var=False`) does not require the groups to have equal variances and is a useful default.

The groups must be independent; if the same people are measured twice, use a paired analysis instead. Always report the difference in means and an interval, not only a p-value.

In [ ]:
rng_ttest = np.random.default_rng(30)  # Create a reproducible random generator.
control = rng_ttest.normal(loc=50, scale=10, size=80)  # Simulate outcomes for the control group.
treatment = rng_ttest.normal(loc=54, scale=12, size=75)  # Simulate outcomes for the treatment group.
result = stats.ttest_ind(treatment, control, equal_var=False)  # Compare independent group means using Welch's t-test.
mean_difference = treatment.mean() - control.mean()  # Calculate the observed treatment-minus-control difference.
se_difference = np.sqrt(treatment.var(ddof=1) / len(treatment) + control.var(ddof=1) / len(control))  # Estimate the standard error of the difference.
degrees = result.df  # Read Welch's approximate degrees of freedom from SciPy's result.
critical_t = stats.t.ppf(0.975, df=degrees)  # Obtain the 95% critical value using those degrees of freedom.
ci = (mean_difference - critical_t * se_difference, mean_difference + critical_t * se_difference)  # Calculate a confidence interval for the mean difference.
print("Control mean:", round(control.mean(), 2))  # Display the control mean.
print("Treatment mean:", round(treatment.mean(), 2))  # Display the treatment mean.
print("Difference (treatment - control):", round(mean_difference, 2))  # Display the observed difference.
print("95% CI for difference:", tuple(round(x, 2) for x in ci))  # Display the interval.
print("Welch t-test p-value:", round(result.pvalue, 4))  # Display the test p-value.

## 19. Paired t-Test

A paired t-test compares two measurements from the same subject or matched unit, such as response time before and after a change. The test operates on within-pair differences.

Do not treat paired observations as independent groups. The relevant assumptions concern the distribution of the differences, not the two raw columns separately.

In [ ]:
before = np.array([10.2, 9.8, 11.0, 10.5, 9.9, 10.7, 10.1, 11.2])  # Record each subject's measurement before a change.
after = np.array([9.7, 9.5, 10.4, 10.0, 9.6, 10.0, 9.8, 10.6])  # Record the same subjects' measurements after the change.
differences = after - before  # Calculate within-subject differences.
result = stats.ttest_rel(after, before)  # Test whether the average paired difference is zero.
print("Mean difference:", round(differences.mean(), 3))  # Display the average change.
print("t statistic:", round(result.statistic, 3))  # Display the paired-test statistic.
print("p-value:", round(result.pvalue, 4))  # Display the two-sided p-value.
print("Negative difference means the measurement decreased after the change.")  # Explain the sign of the paired difference.

## 20. Comparing Proportions for A/B Tests

For conversion experiments, compare proportions rather than treating a binary outcome as a continuous measurement. This example uses a two-proportion z-test formula for independent groups.

The large-sample z-test relies on adequate expected counts and independent users. For small samples or rare outcomes, consider exact methods. Decide whether the test is one- or two-sided before examining results.

In [ ]:
conversions_a, users_a = 45, 500  # Record conversions and total users for variant A.
conversions_b, users_b = 60, 500  # Record conversions and total users for variant B.
rate_a = conversions_a / users_a  # Calculate variant A's conversion rate.
rate_b = conversions_b / users_b  # Calculate variant B's conversion rate.
pooled_rate = (conversions_a + conversions_b) / (users_a + users_b)  # Estimate a pooled conversion rate under the null of equal proportions.
standard_error = np.sqrt(pooled_rate * (1 - pooled_rate) * (1 / users_a + 1 / users_b))  # Calculate the null standard error for the rate difference.
z_statistic = (rate_b - rate_a) / standard_error  # Standardize the observed difference under the null.
p_value = 2 * stats.norm.sf(abs(z_statistic))  # Calculate the two-sided p-value from the standard normal distribution.
print("Variant A conversion:", f"{rate_a:.2%}")  # Display A's observed conversion rate.
print("Variant B conversion:", f"{rate_b:.2%}")  # Display B's observed conversion rate.
print("Difference (B - A):", f"{rate_b - rate_a:.2%}")  # Display the absolute percentage-point difference.
print("z statistic:", round(z_statistic, 3))  # Display the z score.
print("Two-sided p-value:", round(p_value, 4))  # Display the large-sample test result.

## 21. Chi-Square Test of Independence

A chi-square test of independence evaluates whether two categorical variables are associated in a contingency table. It compares observed counts with expected counts under independence.

Use counts rather than percentages as test input. Expected counts should generally not be too small; for sparse tables, consider combining categories when scientifically defensible or using an exact test.

In [ ]:
observed = np.array([[40, 30, 20], [25, 35, 30]])  # Create a 2-by-3 table of observed counts for two customer groups and three channels.
chi2, p_value, degrees_freedom, expected = stats.chi2_contingency(observed)  # Test independence and calculate expected cell counts.
print("Observed counts:\n", observed)  # Display the original contingency table.
print("\nExpected counts under independence:\n", expected.round(2))  # Display expected counts if the variables were independent.
print("\nChi-square statistic:", round(chi2, 3))  # Display the test statistic.
print("Degrees of freedom:", degrees_freedom)  # Display degrees of freedom.
print("p-value:", round(p_value, 4))  # Display evidence against independence under the test assumptions.

## 22. Chi-Square Goodness-of-Fit Test

A chi-square goodness-of-fit test compares observed category counts with a specified expected distribution. For example, it can check whether channel choices align with a predeclared 50/30/20 distribution.

Expected counts should be specified before inspecting the result and generally should be sufficiently large. A rejection says the observed pattern differs from the specified distribution; it does not explain why.

In [ ]:
observed_counts = np.array([120, 80, 100])  # Record observed counts across three categories.
expected_counts = np.array([100, 100, 100])  # Define the expected counts under the null model.
result = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)  # Compare observed counts with the specified expectations.
print("Observed:", observed_counts)  # Display the observed category counts.
print("Expected:", expected_counts)  # Display the expected category counts.
print("Chi-square statistic:", round(result.statistic, 3))  # Display the goodness-of-fit statistic.
print("p-value:", round(result.pvalue, 4))  # Display evidence against the expected distribution.

## 23. One-Way ANOVA

One-way ANOVA tests whether the means of three or more independent groups are all equal under its assumptions. A significant omnibus result means at least one group mean differs, but does not identify which pairs differ.

The classical test assumes independent observations, approximately normal residuals within groups, and equal variances. If variances differ substantially, Welch's ANOVA or robust alternatives may be more suitable.

In [ ]:
rng_anova = np.random.default_rng(44)  # Create a reproducible random generator.
group_a = rng_anova.normal(100, 12, size=35)  # Simulate observations for group A.
group_b = rng_anova.normal(108, 12, size=34)  # Simulate observations for group B.
group_c = rng_anova.normal(116, 12, size=36)  # Simulate observations for group C.
f_statistic, p_value = stats.f_oneway(group_a, group_b, group_c)  # Run one-way ANOVA across the three groups.
print("Group means:", [round(group.mean(), 2) for group in [group_a, group_b, group_c]])  # Show each group's observed average.
print("F statistic:", round(f_statistic, 3))  # Display the ratio-like ANOVA test statistic.
print("p-value:", round(p_value, 4))  # Display evidence against equal population means.

## 24. Effect Size: Cohen's d

A p-value helps assess compatibility with a null model; **effect size** describes the magnitude of a difference. Cohen's d expresses a difference between two means in pooled-standard-deviation units.

A standardized effect size is not automatically small or large in every domain. Interpret it alongside the raw-unit difference, confidence interval, cost, and practical context.

In [ ]:
control = np.array([48, 50, 52, 49, 51, 47, 53, 50], dtype=float)  # Store example control outcomes.
treatment = np.array([52, 54, 53, 55, 51, 56, 54, 52], dtype=float)  # Store example treatment outcomes.
n1, n2 = len(control), len(treatment)  # Count participants in each group.
pooled_variance = ((n1 - 1) * control.var(ddof=1) + (n2 - 1) * treatment.var(ddof=1)) / (n1 + n2 - 2)  # Calculate the pooled sample variance.
pooled_std = np.sqrt(pooled_variance)  # Convert pooled variance to a standard deviation.
cohens_d = (treatment.mean() - control.mean()) / pooled_std  # Standardize the mean difference.
raw_difference = treatment.mean() - control.mean()  # Calculate the unstandardized difference in original units.
print("Raw mean difference:", round(raw_difference, 2))  # Display the practical-unit change.
print("Cohen's d:", round(cohens_d, 2))  # Display the standardized effect size.

## 25. Pearson and Spearman Correlation

Pearson correlation measures linear association between numeric variables. Spearman correlation measures the association between ranked values and can capture monotonic relationships that are not linear.

Both are measures of association, not causation. Outliers can strongly influence Pearson correlation; Spearman is less dependent on the original scale but can still be misleading with confounding or small samples.

In [ ]:
ad_spend = np.array([10, 12, 15, 18, 20, 25, 27, 30], dtype=float)  # Store advertising spend across periods.
sales = np.array([100, 108, 125, 131, 150, 172, 171, 200], dtype=float)  # Store corresponding sales values.
pearson_r, pearson_p = stats.pearsonr(ad_spend, sales)  # Calculate Pearson's linear correlation and its p-value.
spearman_r, spearman_p = stats.spearmanr(ad_spend, sales)  # Calculate Spearman's rank correlation and its p-value.
print("Pearson r:", round(pearson_r, 3), "| p-value:", round(pearson_p, 4))  # Display the linear association result.
print("Spearman rho:", round(spearman_r, 3), "| p-value:", round(spearman_p, 4))  # Display the rank-based association result.
print("Interpretation caution: neither coefficient establishes causation.")  # Reinforce the distinction between association and cause.

## 26. Simple Linear Regression

Simple linear regression models the relationship between a predictor \(x\) and an outcome \(y\) with a line. The slope estimates the average change in \(y\) associated with a one-unit increase in \(x\) under the model.

Review residuals, outliers, linearity, independence, and variance patterns. A high R-squared alone does not prove causality or guarantee good predictions on new data.

In [ ]:
rng_reg = np.random.default_rng(55)  # Create a reproducible random generator.
advertising = np.linspace(10, 100, 60)  # Create 60 advertising-spend values.
sales = 15 + 2.4 * advertising + rng_reg.normal(0, 18, size=len(advertising))  # Create a linear relationship with random noise.
result = stats.linregress(advertising, sales)  # Fit a simple least-squares line and calculate association statistics.
predicted = result.intercept + result.slope * advertising  # Calculate fitted sales values from the estimated line.
residuals = sales - predicted  # Calculate observed minus fitted sales.
fig, ax = plt.subplots(figsize=(7, 4))  # Create a scatter/regression chart.
ax.scatter(advertising, sales, alpha=0.65, label="Observed data")  # Plot the observations.
ax.plot(advertising, predicted, label="Fitted line")  # Draw the fitted regression line.
ax.set_title("Advertising Spend and Sales")  # Add a descriptive title.
ax.set_xlabel("Advertising spend (illustrative units)")  # Label the predictor.
ax.set_ylabel("Sales (illustrative units)")  # Label the outcome.
ax.legend()  # Identify observations and fitted line.
fig.tight_layout()  # Adjust chart spacing.
plt.show()  # Display the relationship.
print("Slope:", round(result.slope, 3))  # Display estimated change in sales per spend unit.
print("R-squared:", round(result.rvalue ** 2, 3))  # Display the squared correlation as R-squared in simple linear regression.
print("Residual standard error (rough):", round(np.sqrt(np.sum(residuals ** 2) / (len(sales) - 2)), 2))  # Estimate residual spread with two fitted parameters.

## 27. Bootstrap Confidence Intervals

Bootstrapping estimates uncertainty by repeatedly sampling observations **with replacement** and recalculating a statistic. It is useful when a closed-form standard error is difficult or when you want an intuitive uncertainty estimate.

Bootstrap quality depends on the sample representing the population and on the sampling unit being correct. For clustered or time-dependent data, resampling individual rows independently may be inappropriate.

In [ ]:
rng_boot = np.random.default_rng(202)  # Create a reproducible generator for bootstrap resampling.
observed = sales_df["revenue"].dropna().to_numpy()  # Extract available revenue observations.
bootstrap_means = np.empty(5000)  # Preallocate storage for 5,000 bootstrap means.
for i in range(len(bootstrap_means)):  # Repeat the resampling process.
    resample = rng_boot.choice(observed, size=len(observed), replace=True)  # Draw a bootstrap sample with replacement.
    bootstrap_means[i] = resample.mean()  # Store the mean of this resample.
ci_lower, ci_upper = np.percentile(bootstrap_means, [2.5, 97.5])  # Use percentile cutoffs for an approximate 95% interval.
print("Observed sample mean:", round(observed.mean(), 2))  # Display the original sample mean.
print("Bootstrap 95% interval:", (round(ci_lower, 2), round(ci_upper, 2)))  # Display the bootstrap interval for the mean.

## 28. Mann–Whitney U Test

The Mann–Whitney U test compares two independent groups using ranks. It is often considered when a t-test's assumptions are questionable, particularly with ordinal data or strongly non-normal distributions.

It is not simply a universal test of medians: its interpretation depends on the shapes of the group distributions. If the distributions differ in shape, a significant result may reflect more than a location shift.

In [ ]:
control = np.array([12, 14, 15, 13, 18, 19, 16, 15, 14])  # Store example response times for a control group.
treatment = np.array([10, 11, 13, 12, 14, 15, 12, 13, 11])  # Store response times for an independent treatment group.
result = stats.mannwhitneyu(treatment, control, alternative="two-sided")  # Compare the groups using their ranks.
print("Control median:", np.median(control))  # Display the control group's median.
print("Treatment median:", np.median(treatment))  # Display the treatment group's median.
print("U statistic:", round(result.statistic, 3))  # Display the Mann–Whitney statistic.
print("p-value:", round(result.pvalue, 4))  # Display the two-sided test p-value.

## 29. Multiple Comparisons and Bonferroni

If many hypotheses are tested, some small p-values can occur by chance even when all null hypotheses are true. The **Bonferroni correction** controls the family-wise error rate by comparing each p-value with `alpha / number_of_tests`, or by multiplying p-values by the number of tests and capping at 1.

Bonferroni is simple and conservative. For many exploratory tests, false-discovery-rate procedures such as Benjamini–Hochberg can be more powerful.

In [ ]:
raw_p_values = np.array([0.01, 0.03, 0.20, 0.049, 0.70])  # Store five illustrative p-values from a family of tests.
alpha = 0.05  # Choose the desired family-wise significance level before interpreting results.
number_of_tests = len(raw_p_values)  # Count tests in the family.
bonferroni_cutoff = alpha / number_of_tests  # Calculate the per-test threshold.
adjusted_p_values = np.minimum(raw_p_values * number_of_tests, 1.0)  # Calculate Bonferroni-adjusted p-values and cap them at one.
significant = adjusted_p_values < alpha  # Mark results significant under the adjusted p-value rule.
print("Bonferroni per-test cutoff:", round(bonferroni_cutoff, 4))  # Display the equivalent per-test threshold.
print("Raw p-values:", raw_p_values)  # Display original p-values.
print("Adjusted p-values:", adjusted_p_values.round(3))  # Display corrected p-values.
print("Significant after correction:", significant)  # Show which tests remain below the chosen family-wise level.

## 30. A/B Test: Practical Interpretation

An A/B test compares an outcome between two variants, ideally using random assignment and a predeclared primary metric. Report the baseline, treatment rate, absolute difference, relative change, uncertainty, and business implications.

Do not stop a test early simply because a p-value becomes small. Check randomization, sample ratio, exposure definitions, duplicate users, seasonality, guardrail metrics, and whether the observed change is large enough to matter commercially.

In [ ]:
users_a, conversions_a = 1200, 108  # Store sample size and conversions for variant A.
users_b, conversions_b = 1180, 124  # Store sample size and conversions for variant B.
rate_a = conversions_a / users_a  # Calculate the baseline conversion rate.
rate_b = conversions_b / users_b  # Calculate the treatment conversion rate.
absolute_change = rate_b - rate_a  # Calculate the absolute change in conversion proportion.
relative_change = absolute_change / rate_a if rate_a != 0 else np.nan  # Calculate relative change while guarding against a zero baseline.
pooled = (conversions_a + conversions_b) / (users_a + users_b)  # Calculate the pooled rate under the equal-proportions null hypothesis.
se_null = np.sqrt(pooled * (1 - pooled) * (1 / users_a + 1 / users_b))  # Estimate the null standard error.
z_score = absolute_change / se_null  # Standardize the observed difference.
p_value = 2 * stats.norm.sf(abs(z_score))  # Compute the two-sided large-sample p-value.
se_diff = np.sqrt(rate_a * (1 - rate_a) / users_a + rate_b * (1 - rate_b) / users_b)  # Estimate the unpooled standard error for an approximate interval.
ci = (absolute_change - 1.96 * se_diff, absolute_change + 1.96 * se_diff)  # Calculate an approximate 95% interval for the absolute difference.
print("Variant A rate:", f"{rate_a:.2%}")  # Display the baseline rate.
print("Variant B rate:", f"{rate_b:.2%}")  # Display the treatment rate.
print("Absolute change:", f"{absolute_change:.2%}")  # Display the percentage-point change.
print("Relative change:", f"{relative_change:.1%}")  # Display relative improvement from baseline.
print("Approx. 95% CI:", (f"{ci[0]:.2%}", f"{ci[1]:.2%}"))  # Display the uncertainty interval for the absolute change.
print("Two-sided p-value:", round(p_value, 4))  # Display the large-sample p-value.
print("Decision should also consider sample quality and business value.")  # Remind the analyst not to decide from the p-value alone.

## 31. Statistical Power and Sample Size Intuition

Statistical power is the probability that a test detects an effect of a specified size when that effect is real. Power depends on the effect size, sample size, variability, significance level, and test design.

Plan sample size before running an experiment. The small simulation below illustrates that, for a fixed effect and variability, larger samples tend to produce narrower uncertainty around the mean difference. It is an illustration—not a formal power calculation.

In [ ]:
rng_power = np.random.default_rng(313)  # Create a reproducible generator for simulated experiments.
for n in [20, 50, 100, 300]:  # Compare increasingly large sample sizes per group.
    control = rng_power.normal(loc=100, scale=15, size=n)  # Generate control outcomes with a fixed mean and spread.
    treatment = rng_power.normal(loc=105, scale=15, size=n)  # Generate treatment outcomes with a five-unit mean difference.
    se = np.sqrt(control.var(ddof=1) / n + treatment.var(ddof=1) / n)  # Estimate the standard error of the difference in means.
    difference = treatment.mean() - control.mean()  # Calculate the observed treatment-minus-control difference.
    ci_width = 2 * 1.96 * se  # Approximate the full width of a 95% interval using a normal critical value.
    print(f"n per group={n:3d} | observed difference={difference:6.2f} | approximate CI width={ci_width:5.2f}")  # Compare estimates and uncertainty.

## 32. Common Statistical Mistakes

Good analysts distinguish statistical evidence from practical importance. Frequent mistakes include:
- Treating correlation as causation.
- Reporting a p-value without an effect size or confidence interval.
- Ignoring multiple testing.
- Using a test that does not match the study design.
- Confusing standard deviation with standard error.
- Removing outliers without investigation.
- Treating a non-significant result as proof of no effect.

Statistics supports judgement; it does not replace checking the data collection process, assumptions, or business context.

In [ ]:
review_checklist = [  # Prepare a quick review checklist for a report.
    "Define the population, outcome, and unit of analysis.",  # Make the question and denominator explicit.
    "Inspect missingness, outliers, and data collection quality.",  # Check problems before testing.
    "Choose a test that matches the design and assumptions.",  # Avoid applying a test by habit.
    "Report effect size and interval alongside the p-value.",  # Present magnitude and uncertainty.
    "Account for multiple comparisons when relevant.",  # Reduce false-positive risk from many tests.
    "Separate association from causation.",  # Avoid causal claims from correlation alone.
    "Explain practical and business implications.",  # Connect the numbers to the decision.
]  # Finish the checklist.
for item in review_checklist:  # Display every item for use before publishing a statistical report.
    print("☐", item)  # Print an unchecked checklist line.